In [3]:
import pandas as pd
import numpy as np
from sqlalchemy import text
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))
from db_connect import get_engine

engine = get_engine()

# Load all tables we need
orders      = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
customers   = pd.read_sql("SELECT * FROM customers", engine)
reviews     = pd.read_sql("SELECT * FROM reviews", engine)

# Parse timestamps
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date','order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Tables loaded:")
print(f"  orders:      {orders.shape}")
print(f"  order_items: {order_items.shape}")
print(f"  customers:   {customers.shape}")
print(f"  reviews:     {reviews.shape}")

Tables loaded:
  orders:      (99441, 8)
  order_items: (112650, 7)
  customers:   (99441, 5)
  reviews:     (99224, 7)


Cell 2 — Q1 Replication: Monthly GMV

In [4]:
# ── PANDAS REPLICATION: Q1 — Monthly GMV & AOV ──────────────────────────────
# SQL used: DATE_TRUNC('month'), SUM, AVG, GROUP BY
# Pandas equivalent: merge → filter → resample/groupby on period

delivered = orders[orders['order_status'] == 'delivered']
merged = delivered.merge(order_items, on='order_id')

merged['month'] = merged['order_purchase_timestamp'].dt.to_period('M')

monthly_gmv = merged.groupby('month').agg(
    total_orders=('order_id', 'nunique'),
    gmv=('price', 'sum'),
    aov=('price', 'mean')
).round(2)

print(monthly_gmv.tail(10))

         total_orders        gmv     aov
month                                   
2017-11          7289  987765.37  116.55
2017-12          5513  726033.19  117.35
2018-01          7069  924645.00  115.05
2018-02          6555  826437.13  109.93
2018-03          7003  953356.25  118.92
2018-04          6798  973534.09  124.38
2018-05          6749  977544.69  125.17
2018-06          6099  856077.86  122.12
2018-07          6159  867953.46  124.65
2018-08          6351  838576.64  117.41


Cell 3 — Q7 Replication: Delivery Delay vs Review Score


In [5]:
# ── PANDAS REPLICATION: Q7 — Review Score vs Delivery Delay ─────────────────
# SQL used: CASE WHEN with EXTRACT(EPOCH), JOIN 3 tables
# Pandas equivalent: merge → timedelta → pd.cut → groupby

df = delivered.merge(reviews, on='order_id')
df = df.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])

df['delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.total_seconds() / 86400

bins   = [-999, -5, 0, 5, 999]
labels = ['Very Early (>5d)', 'Early', 'Late (1-5d)', 'Very Late (>5d)']
df['delivery_bucket'] = pd.cut(df['delay_days'], bins=bins, labels=labels)

result = df.groupby('delivery_bucket', observed=True).agg(
    total_orders=('order_id', 'count'),
    avg_review_score=('review_score', 'mean')
).round(2)

print(result)

                  total_orders  avg_review_score
delivery_bucket                                 
Very Early (>5d)         79791              4.31
Early                     8862              4.15
Late (1-5d)               3584              3.46
Very Late (>5d)           4116              1.79


Cell 4 — Q8 Replication: Repeat Customer Rate


In [6]:
# ── PANDAS REPLICATION: Q8 — Repeat Customer Rate ───────────────────────────
# SQL used: CTE → COUNT DISTINCT → CASE WHEN → GROUP BY
# Pandas equivalent: merge → groupby → map → value_counts

cust_orders = customers.merge(orders, on='customer_id')

order_counts = cust_orders.groupby('customer_unique_id')['order_id'].nunique().reset_index()
order_counts.columns = ['customer_unique_id', 'total_orders']

def classify(n):
    if n == 1: return 'One-time'
    if n == 2: return 'Returned once'
    return 'Loyal (3+ orders)'

order_counts['customer_type'] = order_counts['total_orders'].map(classify)

result = order_counts['customer_type'].value_counts().reset_index()
result.columns = ['customer_type', 'customer_count']
result['pct_of_customers'] = (result['customer_count'] / result['customer_count'].sum() * 100).round(2)

print(result)

       customer_type  customer_count  pct_of_customers
0           One-time           93099             96.88
1      Returned once            2745              2.86
2  Loyal (3+ orders)             252              0.26


Cell 5 — Q11 Replication: Running Total GMV


In [7]:
# ── PANDAS REPLICATION: Q11 — Running Total GMV & MoM Growth ────────────────
# SQL used: CTE → SUM() OVER (ORDER BY) → LAG() → NULLIF()
# Pandas equivalent: groupby → cumsum() → pct_change()

merged['month'] = merged['order_purchase_timestamp'].dt.to_period('M')

monthly = merged.groupby('month').agg(
    monthly_gmv=('price', 'sum')
).round(2)

monthly['cumulative_gmv'] = monthly['monthly_gmv'].cumsum().round(2)
monthly['mom_growth_pct'] = (monthly['monthly_gmv'].pct_change() * 100).round(2)

print(monthly)

         monthly_gmv  cumulative_gmv  mom_growth_pct
month                                               
2016-09       134.97          134.97             NaN
2016-10     40325.11        40460.08        29777.09
2016-12        10.90        40470.98          -99.97
2017-01    111798.36       152269.34      1025573.03
2017-02    234223.40       386492.74          109.51
2017-03    359198.85       745691.59           53.36
2017-04    340669.68      1086361.27           -5.16
2017-05    489338.25      1575699.52           43.64
2017-06    421923.37      1997622.89          -13.78
2017-07    481604.52      2479227.41           14.15
2017-08    554699.70      3033927.11           15.18
2017-09    607399.67      3641326.78            9.50
2017-10    648247.65      4289574.43            6.73
2017-11    987765.37      5277339.80           52.37
2017-12    726033.19      6003372.99          -26.50
2018-01    924645.00      6928017.99           27.36
2018-02    826437.13      7754455.12          

Cell 6 — T-test: Do SP customers rate differently than non-SP?


In [8]:
# ── HYPOTHESIS TEST 1: T-test ────────────────────────────────────────────────
# Question: Do customers in SP state give significantly different review scores
#           than customers in other states?
# H0: Mean review score in SP == Mean review score outside SP
# H1: They are significantly different (two-tailed)

from scipy import stats

df_reviews = orders.merge(reviews, on='order_id').merge(customers, on='customer_id')
df_reviews = df_reviews.dropna(subset=['review_score', 'customer_state'])

sp_scores    = df_reviews[df_reviews['customer_state'] == 'SP']['review_score']
non_sp_scores = df_reviews[df_reviews['customer_state'] != 'SP']['review_score']

t_stat, p_value = stats.ttest_ind(sp_scores, non_sp_scores)

print(f"SP mean score:     {sp_scores.mean():.4f}  (n={len(sp_scores):,})")
print(f"Non-SP mean score: {non_sp_scores.mean():.4f}  (n={len(non_sp_scores):,})")
print(f"T-statistic:       {t_stat:.4f}")
print(f"P-value:           {p_value:.6f}")
print()
if p_value < 0.05:
    print("Result: REJECT H0 — statistically significant difference (p < 0.05)")
else:
    print("Result: FAIL TO REJECT H0 — no significant difference (p >= 0.05)")

SP mean score:     4.1740  (n=41,690)
Non-SP mean score: 4.0230  (n=57,534)
T-statistic:       17.4432
P-value:           0.000000

Result: REJECT H0 — statistically significant difference (p < 0.05)


Cell 7 — Chi-square: Is payment type independent of customer state?


In [9]:
# ── HYPOTHESIS TEST 2: Chi-Square Test ──────────────────────────────────────
# Question: Is payment type choice independent of customer state?
# H0: Payment type and customer state are independent
# H1: There is a significant association between them
# Using top 5 states and top 3 payment types to keep contingency table clean

from scipy.stats import chi2_contingency
import pandas as pd

df_pay = orders.merge(
    pd.read_sql("SELECT * FROM payments", engine), on='order_id'
).merge(customers, on='customer_id')

top_states   = ['SP', 'RJ', 'MG', 'RS', 'PR']
top_payments = ['credit_card', 'boleto', 'debit_card']

df_filtered = df_pay[
    df_pay['customer_state'].isin(top_states) &
    df_pay['payment_type'].isin(top_payments)
]

contingency = pd.crosstab(df_filtered['customer_state'], df_filtered['payment_type'])
print("Contingency Table:")
print(contingency)
print()

chi2, p, dof, expected = chi2_contingency(contingency)
print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom:   {dof}")
print(f"P-value:              {p:.6f}")
print()
if p < 0.05:
    print("Result: REJECT H0 — payment type and state are NOT independent (p < 0.05)")
else:
    print("Result: FAIL TO REJECT H0 — no significant association (p >= 0.05)")

Contingency Table:
payment_type    boleto  credit_card  debit_card
customer_state                                 
MG                2304         9070         139
PR                1118         3786          75
RJ                2163        10288         185
RS                1359         3985          76
SP                8205        32168         759

Chi-square statistic: 200.8694
Degrees of freedom:   8
P-value:              0.000000

Result: REJECT H0 — payment type and state are NOT independent (p < 0.05)


In [10]:
# ── STAGE 5: LINEAR TREND FORECASTING — Next Month GMV ──────────────────────
# Using monthly GMV from Q11 replication
# Model: OLS linear regression on time index → forecast next 3 months

import numpy as np
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Use the monthly df from Cell 5, drop first month (incomplete) and NaN
forecast_df = monthly.reset_index().dropna()
forecast_df = forecast_df[forecast_df['monthly_gmv'] > 1000].copy()
forecast_df['t'] = np.arange(len(forecast_df))

X = forecast_df[['t']]
y = forecast_df['monthly_gmv']

model = LinearRegression()
model.fit(X, y)

r2 = model.score(X, y)
print(f"Model R²: {r2:.4f}")
print(f"Slope:    R${model.coef_[0]:,.2f} GMV per month")
print(f"Intercept: R${model.intercept_:,.2f}")
print()

# Forecast next 3 months beyond the dataset
last_t = forecast_df['t'].max()
last_month = forecast_df['month'].max()

print("Forecast:")
for i in range(1, 4):
    t_future = last_t + i
    gmv_pred = model.predict([[t_future]])[0]
    future_month = last_month + i
    print(f"  {future_month}: R${gmv_pred:,.2f}")

Model R²: 0.8380
Slope:    R$43,998.25 GMV per month
Intercept: R$189,605.69

Forecast:
  2018-09: R$1,113,568.96
  2018-10: R$1,157,567.22
  2018-11: R$1,201,565.47
